imports

In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller

Data loading

In [2]:
df=pd.read_csv("D:\Delphi Forcasting\consumer_health_products_timeseries.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1102 entries, 0 to 1101
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   City                  1102 non-null   object
 1   Category              1102 non-null   object
 2   Brand                 1102 non-null   object
 3   Sub Brand             1102 non-null   object
 4   Product Name          1102 non-null   object
 5   Month                 1102 non-null   object
 6   Volume Sales (units)  1102 non-null   int64 
dtypes: int64(1), object(6)
memory usage: 60.4+ KB


In [3]:
df.isna().sum()
df.duplicated().sum()

np.int64(0)

Droup Duplicate

In [4]:
df= df.drop_duplicates()
df=df[df["Volume Sales (units)"]>0]

#Missing Month

In [5]:
all_months = pd.date_range(df["Month"].min(), df["Month"].max(), freq='MS')

def make_continuous(ts_df):
    ts_df = ts_df.set_index("Month").reindex(all_months)
    ts_df["Volume Sales (units)"] = ts_df["Volume Sales (units)"].interpolate(method='linear')
    ts_df = ts_df.reset_index().rename(columns={"index": "Month"})
    return ts_df

continuous_df = (
    df.groupby("Product Name", group_keys=False)
    .apply(make_continuous)
)


C:\Users\S Vishnu\AppData\Local\Temp\ipykernel_23508\3321550791.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(make_continuous)


In [7]:
def remove_outliers(ts_df):
    Q1 = ts_df["Volume Sales (units)"].quantile(0.25)
    Q3 = ts_df["Volume Sales (units)"].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    ts_df["Volume Sales (units)"] = ts_df["Volume Sales (units)"].clip(lower, upper)
    return ts_df

clean_df = (
    continuous_df.groupby("Product Name", group_keys=False)
    .apply(remove_outliers)
)

C:\Users\S Vishnu\AppData\Local\Temp\ipykernel_23508\2134108703.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(remove_outliers)


In [8]:
def adf_test(series):
    result = adfuller(series.dropna())
    return {"ADF Statistic": result[0], "p-value": result[1]}

clean_df = clean_df.reset_index(drop=True)

stationarity_results = (
    clean_df.groupby("Product Name")["Volume Sales (units)"]
    .apply(adf_test).apply(pd.Series)
)

In [14]:
def segment_demand(ts_df):
    series = ts_df["Volume Sales (units)"]
    mean, std = series.mean(), series.std()
    cv = std / mean if mean != 0 else 0

    autocorr = series.autocorr(lag=12)
    trend = np.polyfit(range(len(series)), series, 1)[0]

    if cv < 0.2:
        return "Stable"
    elif abs(trend) > 20:
        return "Trend"
    elif autocorr > 0.5:
        return "Seasonal"
    else:
        return "Intermittent"

# ✅ This always returns a clean DataFrame
segmentation = (
    clean_df.groupby("Product Name", as_index=False)
    .apply(lambda x: pd.Series({"Demand Pattern": segment_demand(x)}))
)

print(segmentation.head())


Empty DataFrame
Columns: [Month, City, Category, Brand, Sub Brand, Product Name, Volume Sales (units)]
Index: []


C:\Users\S Vishnu\AppData\Local\Temp\ipykernel_23508\3218376502.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({"Demand Pattern": segment_demand(x)}))


In [15]:
summary = (
    clean_df.groupby("Product Name")
    .agg(
        Start_Date=("Month", "min"),
        End_Date=("Month", "max"),
        Length=("Month", "count"),
        Avg_Sales=("Volume Sales (units)", "mean"),
        Std_Sales=("Volume Sales (units)", "std")
    )
    .reset_index()
)

In [20]:
segmentation = segmentation.reset_index(drop=True)
if 'Product Name' in segmentation.index.names:
    segmentation = segmentation.reset_index()
segmentation = segmentation.loc[:, ~segmentation.columns.duplicated()]

print("Segmentation preview:")
print(segmentation.head())

# ✅ Clean merge
summary = summary.reset_index(drop=True)
report = summary.merge(segmentation, on="Product Name", how="left")

report.to_csv("time_series_segmentation_report.csv", index=False)
print("\n✅ Report generated successfully!")
print(report.head())

Segmentation preview:
Empty DataFrame
Columns: [Month, City, Category, Brand, Sub Brand, Product Name, Volume Sales (units)]
Index: []

✅ Report generated successfully!
Empty DataFrame
Columns: [Product Name, Start_Date, End_Date, Length, Avg_Sales, Std_Sales, Month, City, Category, Brand, Sub Brand, Volume Sales (units)]
Index: []


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

C:\Users\S Vishnu\AppData\Local\Temp\ipykernel_23508\2806127714.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(segment_demand)


In [ ]:
# -----------------------------
# 1️⃣ Load your cleaned dataset
# -----------------------------

clean_df = pd.read_csv("D:\Delphi Forcasting\consumer_health_products_timeseries.csv")

In [ ]:
# -----------------------------
# 2️⃣ Group & summarize each product
# -----------------------------
summary = (
    clean_df.groupby("Product Name")
    .agg(
        Start_Date=("Month", "min"),
        End_Date=("Month", "max"),
        Length=("Month", "count"),
        Avg_Sales=("Volume Sales (units)", "mean"),
        Std_Sales=("Volume Sales (units)", "std")
    )
    .reset_index()
)

In [ ]:
# -----------------------------
# 3️⃣ Define segmentation logic
# -----------------------------
def segment_demand(product_df):
    mean_sales = product_df["Volume Sales (units)"].mean()
    std_sales = product_df["Volume Sales (units)"].std()
    
    if std_sales < 0.1 * mean_sales:
        return "Stable"
    elif std_sales < 0.3 * mean_sales:
        return "Fluctuating"
    else:
        return "Intermittent"

In [ ]:
    # -----------------------------
# 4️⃣ Apply segmentation per product
# -----------------------------
segmentation = (
    clean_df.groupby("Product Name")
    .apply(segment_demand)
    .reset_index(name="Demand Pattern")
)

In [ ]:
# -----------------------------
# 5️⃣ Merge summary + segmentation
# -----------------------------
report = pd.merge(summary, segmentation, on="Product Name", how="left")

In [ ]:
# -----------------------------
# 6️⃣ Export final CSV report
# -----------------------------
report.to_csv("time_series_segmentation_report.csv", index=False)

In [ ]:
corr_data = clean_df.pivot_table(
    index='Month',
    columns='Product Name',
    values='Volume Sales (units)',
    aggfunc='sum'
)

# Compute correlation
corr_matrix = corr_data.corr()  # Pearson correlation by default

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Heatmap of Product Sales")
plt.show()

In [ ]:
subset_products = ["Product_1", "Product_2", "Product_3", "Product_4"]
subset_corr = corr_data[subset_products].corr()
sns.heatmap(subset_corr, annot=True, cmap="coolwarm")